generator libs

In [1]:
from dataset_generators.valid_bpd_datasets import(
 generate_adversarial_dataset,
 generate_bnpl_dataset,
 generate_calibration_dataset,
 generate_churn_dataset,
 generate_credit_scoring_dataset,
 generate_delivery_failure_dataset,
 generate_disparate_impact_dataset,
 generate_employee_attrition_dataset,
 generate_equalized_odds_dataset,
 generate_fraud_dataset,
 generate_insurance_claim_dataset,
 generate_medical_diagnosis_dataset,
 generate_phishing_url_dataset,
 generate_predictive_maintenance_dataset,
 generate_proxy_dataset,
 generate_purchase_dataset,
 generate_recruitment_dataset,
 generate_spam_dataset,
 generate_ng_fintech_dataset
)
from dataset_generator_utils import DatasetBundle

Dataset generators

In [2]:
DATASET_GENERATORS = {
    "fraud": generate_fraud_dataset,
    "credit_scoring": generate_credit_scoring_dataset,
    "churn": generate_churn_dataset,
    "insurance": generate_insurance_claim_dataset,
    "medical": generate_medical_diagnosis_dataset,
    "attrition": generate_employee_attrition_dataset,
    "purchase": generate_purchase_dataset,
    "recruitment": generate_recruitment_dataset,
    "spam": generate_spam_dataset,
    "phishing": generate_phishing_url_dataset,
    "maintenance": generate_predictive_maintenance_dataset,
    "delivery": generate_delivery_failure_dataset,
    "bnpl": generate_bnpl_dataset,
    "ng_fintech": generate_ng_fintech_dataset,
    "proxy": generate_proxy_dataset,
    "disparate_impact": generate_disparate_impact_dataset,
    "equalized_odds": generate_equalized_odds_dataset,
    "calibration": generate_calibration_dataset,
    "adversarial": generate_adversarial_dataset,
}

# storing the generated bundles to avoid regenerating them
data_bundles = {}

training the model

In [3]:
import os
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from dataset_generator_utils import train_test_validation_split
# model directory
MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

def train_model(bundle: DatasetBundle, name: str) -> dict:
    """
    Train a logistic regression model with train/val/test split.
    Save the trained model to disk and return metrics.
    """
    data_split = train_test_validation_split(bundle)

    X_train, y_train = data_split['train'].X, data_split['train'].y
    X_val, y_val = data_split['validation'].X, data_split['validation'].y
    X_test, y_test = data_split["test"].X, data_split["test"].y

    model = LogisticRegression(max_iter=500)
    model.fit(X_train, y_train)

    preds_val = model.predict(X_val)
    preds_test = model.predict(X_test)

    val_acc = accuracy_score(y_val, preds_val)
    test_acc = accuracy_score(y_test, preds_test)

    # Save model
    model_path = os.path.join(MODEL_DIR, f"{name}_model.pkl")
    joblib.dump(model, model_path)

    return {
        "dataset": name,
        "train_rows": len(X_train),
        "val_rows": len(X_val),
        "test_rows": len(X_test),
        "features": list(X_train.columns),
        "val_accuracy": val_acc,
        "test_accuracy": test_acc,
        "model_path": model_path,
        "metadata": bundle.metadata,
    }


In [4]:
import os
import pandas as pd
import multiprocessing
from concurrent.futures import ThreadPoolExecutor, as_completed

def optimal_workers(num_urls: int) -> int:
    """
    Dynamically choose a safe number of workers.
    - At least 2
    - At most CPU count * 2
    - Never more than number of URLs
    """
    # Get the number of CPUs available
    cpu_count = os.cpu_count() or multiprocessing.cpu_count() or 4
    return max(2, min(num_urls, cpu_count * 2))


def train_all_datasets():
    """
    Train models for all datasets in DATASET_GENERATORS using ThreadPoolExecutor.
    Returns a DataFrame with training results.
    """
    results = []

    max_workers = optimal_workers(len(DATASET_GENERATORS))
    print(f"\nUsing {max_workers} workers for {len(DATASET_GENERATORS)} datasets...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:

         # Generate bundles first and store them
        bundles = {name: gen() for name, gen in DATASET_GENERATORS.items()}
        data_bundles.update(bundles)

        # Submit training jobs
        future_to_name = {
            executor.submit(train_model, bundle, name): name
            for name, bundle in bundles.items()
        }

        # Process completed futures and collect results
        for number, future in enumerate(as_completed(future_to_name), start=1):
            name = future_to_name[future]
            print(f"\n[{number}/{len(future_to_name)}] Training on {name} dataset...")
            try:
                result = future.result()
                results.append(result)
                print(f"{name} done | Val Acc: {result['val_accuracy']:.3f} | Test Acc: {result['test_accuracy']:.3f}")
                print(f"Model saved at {result['model_path']}")
            except Exception as e:
                print(f"{name} failed: {e}")
                results.append({"dataset": name, "status": "error", "error": str(e)})

    return pd.DataFrame(results)

Generate Bdp reports

In [5]:
from bdp_model_gate import ModelGate, StructuredGateContext
from sklearn.model_selection import train_test_split
import joblib
import os

gate = ModelGate()
def generate_model_gate_reports(bundles: dict = None):
    """
    Generate ModelGate reports for all datasets using saved models.
    """
    if bundles is None:
        bundles = {name: gen() for name, gen in DATASET_GENERATORS.items()}

    # Load models into a dict keyed by dataset name
    models = {
        name: joblib.load(f"saved_models/{name}_model.pkl")
        for name in bundles.keys()
        if os.path.exists(f"saved_models/{name}_model.pkl")
    }

    max_workers = optimal_workers(len(bundles))
    print(f"\nUsing {max_workers} workers for {len(bundles)} datasets...")

    reports = {}
    # Use ThreadPoolExecutor to run gate.run in parallel for each dataset
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_name = {}

        for name, bundle in bundles.items():
            data_split = train_test_validation_split(bundle)

            model = models.get(name)
            if model is None:
                print(f"No saved model found for {name}, skipping...")
                continue

            # Build context depending on whether protected_df is present
            if bundle.protected_df is None:
                ctx = StructuredGateContext(
                    model =model, 
                    X=data_split['validation'].X, 
                    y_true=data_split['validation'].y.tolist(),
                    y_pred=model.predict(data_split['validation'].X))
            else:
                # Split the protected_df to get the validation portion
                _, prot_val = train_test_split(bundle.protected_df, test_size=0.2, random_state=42)

                ctx = StructuredGateContext(
                    model =model,
                    X=data_split['validation'].X,
                    y_true=data_split['validation'].y.tolist(),
                    y_pred=model.predict(data_split['validation'].X),
                    protected_df=prot_val)

            future = executor.submit(gate.run, ctx)
            future_to_name[future] = name

        # Process completed futures and collect reports
        for number, future in enumerate(as_completed(future_to_name), start=1):
            name = future_to_name[future]
            print(f"\n[{number}/{len(future_to_name)}] Generating report for {name} dataset...")
            try:
                report = future.result()
                reports[name] = report
                print(f"{name} report generated successfully.")
            except Exception as e:
                print(f"{name} report generation failed: {e}")
                reports[name] = {"status": "error", "error": str(e)}

    return reports


In [6]:
# Example usage:
results = train_all_datasets()


Using 4 workers for 19 datasets...



[1/19] Training on churn dataset...
churn done | Val Acc: 0.980 | Test Acc: 0.982
Model saved at saved_models/churn_model.pkl

[2/19] Training on medical dataset...
medical done | Val Acc: 0.856 | Test Acc: 0.892
Model saved at saved_models/medical_model.pkl

[3/19] Training on insurance dataset...
insurance done | Val Acc: 0.634 | Test Acc: 0.613
Model saved at saved_models/insurance_model.pkl


/home/codespace/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



[4/19] Training on fraud dataset...
fraud done | Val Acc: 0.975 | Test Acc: 0.973
Model saved at saved_models/fraud_model.pkl


/home/codespace/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



[5/19] Training on credit_scoring dataset...
credit_scoring done | Val Acc: 0.798 | Test Acc: 0.798
Model saved at saved_models/credit_scoring_model.pkl

[6/19] Training on spam dataset...
spam failed: could not convert string to float: 'Your appointment is confirmed thanks'

[7/19] Training on phishing dataset...
phishing failed: could not convert string to float: 'https://amazon.com/home'

[8/19] Training on maintenance dataset...
maintenance failed: The DType <class 'numpy.dtypes.DateTime64DType'> could not be promoted by <class 'numpy.dtypes.Float64DType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'

/home/codespace/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



[11/19] Training on attrition dataset...
attrition done | Val Acc: 0.993 | Test Acc: 0.998
Model saved at saved_models/attrition_model.pkl

[12/19] Training on proxy dataset...
proxy done | Val Acc: 0.682 | Test Acc: 0.727
Model saved at saved_models/proxy_model.pkl

[13/19] Training on bnpl dataset...
bnpl done | Val Acc: 0.960 | Test Acc: 0.958
Model saved at saved_models/bnpl_model.pkl

[14/19] Training on disparate_impact dataset...
disparate_impact done | Val Acc: 0.713 | Test Acc: 0.687
Model saved at saved_models/disparate_impact_model.pkl

[15/19] Training on equalized_odds dataset...
equalized_odds done | Val Acc: 0.809 | Test Acc: 0.796
Model saved at saved_models/equalized_odds_model.pkl

[16/19] Training on calibration dataset...
calibration done | Val Acc: 0.744 | Test Acc: 0.723
Model saved at saved_models/calibration_model.pkl

[17/19] Training on adversarial dataset...
adversarial done | Val Acc: 1.000 | Test Acc: 1.000
Model saved at saved_models/adversarial_model.pkl

/home/codespace/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/codespace/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.h

In [7]:
reports = generate_model_gate_reports(data_bundles)


Using 4 workers for 19 datasets...
No saved model found for spam, skipping...
No saved model found for phishing, skipping...
No saved model found for maintenance, skipping...

[1/16] Generating report for bnpl dataset...
bnpl report generation failed: context.protected_df has 800 rows, but context.X has 600 rows — they must be row-aligned

[2/16] Generating report for medical dataset...
medical report generation failed: context.protected_df has 1000 rows, but context.X has 750 rows — they must be row-aligned

[3/16] Generating report for disparate_impact dataset...
disparate_impact report generation failed: context.protected_df has 800 rows, but context.X has 600 rows — they must be row-aligned

[4/16] Generating report for delivery dataset...
delivery report generation failed: context.protected_df has 1000 rows, but context.X has 750 rows — they must be row-aligned

[5/16] Generating report for credit_scoring dataset...
credit_scoring report generation failed: context.protected_df ha

In [8]:
print(f"\nreports generated: {reports.keys()}")
for keys, values in reports.items():
    print(f"\nReport for {keys}:")
    print(values)


reports generated: dict_keys(['bnpl', 'medical', 'disparate_impact', 'delivery', 'credit_scoring', 'ng_fintech', 'attrition', 'recruitment', 'proxy', 'equalized_odds', 'insurance', 'churn', 'purchase', 'fraud', 'calibration', 'adversarial'])

Report for bnpl:
{'status': 'error', 'error': 'context.protected_df has 800 rows, but context.X has 600 rows — they must be row-aligned'}

Report for medical:
{'status': 'error', 'error': 'context.protected_df has 1000 rows, but context.X has 750 rows — they must be row-aligned'}

Report for disparate_impact:
{'status': 'error', 'error': 'context.protected_df has 800 rows, but context.X has 600 rows — they must be row-aligned'}

Report for delivery:
{'status': 'error', 'error': 'context.protected_df has 1000 rows, but context.X has 750 rows — they must be row-aligned'}

Report for credit_scoring:
{'status': 'error', 'error': 'context.protected_df has 600 rows, but context.X has 450 rows — they must be row-aligned'}

Report for ng_fintech:
{'statu

pretty print

In [9]:
import pprint

def pretty_print_reports(reports: dict):
    """
    Pretty print ModelGate reports dictionary in a readable format.
    """
    print("\n=== ModelGate Reports Summary ===\n")
    for name, report in reports.items():
        print(f"Report for: {name} dataset")
        if isinstance(report, dict) and "status" in report and report["status"] == "error":
            print(f"Error: {report['error']}\n")
        else:
            # GateReport objects often have attributes like model_score, model_metric, task, results
            try:
                print(f"Status: OK")
                if hasattr(report, "model_score"):
                    print(f"Model Score: {report.model_score}")
                if hasattr(report, "model_metric"):
                    print(f"Metric: {report.model_metric}")
                if hasattr(report, "task"):
                    print(f"Task: {report.task}")

                # Print a few key checks
                print("Checks:")
                for res in report.results[:5]:  # show first 5 checks for brevity
                    print(f"     - {res.check_name}: {res.flag} ({res.category})")
            except Exception:
                # Fallback if report is just a dict or string
                pprint.pprint(report)
            print("\n")



In [11]:
pretty_print_reports(reports)


=== ModelGate Reports Summary ===

Report for: bnpl dataset
Error: context.protected_df has 800 rows, but context.X has 600 rows — they must be row-aligned

Report for: medical dataset
Error: context.protected_df has 1000 rows, but context.X has 750 rows — they must be row-aligned

Report for: disparate_impact dataset
Error: context.protected_df has 800 rows, but context.X has 600 rows — they must be row-aligned

Report for: delivery dataset
Error: context.protected_df has 1000 rows, but context.X has 750 rows — they must be row-aligned

Report for: credit_scoring dataset
Error: context.protected_df has 600 rows, but context.X has 450 rows — they must be row-aligned

Report for: ng_fintech dataset
Error: context.protected_df has 1000 rows, but context.X has 750 rows — they must be row-aligned

Report for: attrition dataset
Error: context.protected_df has 600 rows, but context.X has 450 rows — they must be row-aligned

Report for: recruitment dataset
Error: context.protected_df has 600